In [1]:
# titanic_compare_cv_pro.py
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from typing import Dict, List, Tuple, Optional

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, train_test_split
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

# ----------------------------- Config ---------------------------------
SEED = 42
OUTER_N_SPLITS = 5
OUTER_N_REPEATS = 3

# Feature selection (set one or none)
TOP_K: Optional[int] = None        # e.g., 7; if None, keep all
CUM_IMPORTANCE: Optional[float] = None  # e.g., 0.90; used only if TOP_K is None

# RF tuning (inner CV)
RF_PARAM_GRID = {
    "max_depth": [3, 5, 7, 9, 12, None],
    "n_estimators": [200, 300, 500],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", "log2"],
}
RF_CALIBRATE = False   # True = calibrate with isotonic (cv=3) inside each outer fold / final fit

# MLP training
MLP_EPOCHS = 35
MLP_LR = 1e-3
MLP_PATIENCE = 7
MLP_BATCH = 256

# Paths
DEFAULT_TRAIN = "data/1.titanic/train.csv"
DEFAULT_TEST  = "data/1.titanic/test.csv"
OUTPUT_DIR    = "."

# ----------------------------------------------------------------------
torch.manual_seed(SEED); np.random.seed(SEED)


# ========================= Preprocessing ===============================
def build_features(df: pd.DataFrame, fit_stats: Optional[Dict]=None):
    """Case-insensitive, leakage-safe feature builder."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    y = None
    if 'survived' in df.columns:
        y = df['survived'].astype(np.float32).values

    if fit_stats is None:
        fit_stats = {}
        fit_stats['age_mean']  = df['age'].mean()  if 'age'  in df.columns else 30.0
        fit_stats['fare_mean'] = df['fare'].mean() if 'fare' in df.columns else 32.2
        if 'embarked' in df.columns and not df['embarked'].dropna().empty:
            fit_stats['embarked_mode'] = df['embarked'].dropna().mode().iloc[0]
        else:
            fit_stats['embarked_mode'] = 'S'

    # Impute
    df['age']  = (df['age']  if 'age'  in df.columns else np.nan).fillna(fit_stats['age_mean'])
    df['fare'] = (df['fare'] if 'fare' in df.columns else np.nan).fillna(fit_stats['fare_mean'])
    df['embarked'] = (df['embarked'] if 'embarked' in df.columns else np.nan).fillna(fit_stats['embarked_mode'])

    # Encodings
    df['sex_bin'] = (df['sex'].map({'male':1.0,'female':0.0}) if 'sex' in df.columns else 1.0)
    df['sex_bin'] = df['sex_bin'].fillna(1.0).astype(float)

    if 'pclass' in df.columns:
        for k in [1,2,3]:
            df[f'pclass_{k}'] = (df['pclass'] == k).astype(float)
    else:
        df['pclass_1']=0.0; df['pclass_2']=0.0; df['pclass_3']=1.0

    df['embarked'] = df['embarked'].astype(str).str.upper().str[0]
    for v in ['C','Q','S']:
        df[f'embarked_{v}'] = (df['embarked'] == v).astype(float)

    # Derived
    df['sibsp'] = df['sibsp'] if 'sibsp' in df.columns else 0
    df['parch'] = df['parch'] if 'parch' in df.columns else 0
    df['family_size'] = df['sibsp'].astype(float) + df['parch'].astype(float) + 1.0
    df['is_alone'] = (df['family_size'] == 1).astype(float)
    df['fare_per_person'] = df['fare'] / df['family_size'].replace(0,1)

    # Cabin signal
    if 'cabin' in df.columns:
        df['has_cabin'] = (~df['cabin'].isna()) & (df['cabin'].astype(str).str.strip() != '')
        df['has_cabin'] = df['has_cabin'].astype(float)
    else:
        df['has_cabin'] = 0.0

    # Scale numerics (fit on train only)
    num_cols = ['age','sibsp','parch','fare','family_size','fare_per_person']
    bin_cols = ['sex_bin','is_alone','has_cabin']
    oh_cols  = [f'pclass_{k}' for k in [1,2,3]] + [f'embarked_{v}' for v in ['C','Q','S']]

    if 'scaler_means' not in (fit_stats or {}):
        fit_stats['scaler_means'] = {c: df[c].astype(float).mean() for c in num_cols}
        fit_stats['scaler_stds']  = {c: (df[c].astype(float).std(ddof=0) or 1.0) for c in num_cols}

    for c in num_cols:
        m = fit_stats['scaler_means'][c]; s = fit_stats['scaler_stds'][c] or 1.0
        df[c] = (df[c].astype(float) - m) / s

    feature_cols = num_cols + bin_cols + oh_cols
    X = df[feature_cols].astype(np.float32).values
    meta = {'feature_cols': feature_cols, 'fit_stats': fit_stats}
    return X, y, meta


def features_for_indices(df: pd.DataFrame, train_idx, val_idx):
    """Leakage-safe: fit stats on train fold, apply to val fold."""
    df_tr = df.iloc[train_idx]
    df_va = df.iloc[val_idx]
    X_tr, y_tr, meta_tr = build_features(df_tr, fit_stats=None)
    X_va, y_va, _       = build_features(df_va, fit_stats=meta_tr['fit_stats'])
    feat_names = np.array(meta_tr['feature_cols'])
    return X_tr, y_tr, X_va, y_va, feat_names


def align_to_train_columns(X_new: np.ndarray, cols_new: List[str], cols_train: List[str]) -> np.ndarray:
    """Ensure X_new matches the order and presence of cols_train (fill missing with 0)."""
    col_to_idx = {c:i for i, c in enumerate(cols_new)}
    aligned = np.zeros((X_new.shape[0], len(cols_train)), dtype=X_new.dtype)
    for j, c in enumerate(cols_train):
        if c in col_to_idx:
            aligned[:, j] = X_new[:, col_to_idx[c]]
        else:
            aligned[:, j] = 0.0
    return aligned


# ===================== Models & Training ===============================
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(64, 32), pdrop=0.2):
        super().__init__()
        layers, last = [], in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(pdrop)]
            last = h
        layers += [nn.Linear(last, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(1)

def train_mlp(X_train, y_train, X_valid, y_valid,
              epochs=MLP_EPOCHS, lr=MLP_LR, patience=MLP_PATIENCE,
              batch_size=MLP_BATCH, device=None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = MLP(in_dim=X_train.shape[1], hidden=(64,32), pdrop=0.2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()

    trn = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=batch_size, shuffle=True)
    val = DataLoader(TensorDataset(torch.tensor(X_valid), torch.tensor(y_valid)), batch_size=batch_size, shuffle=False)

    best, best_val, no_imp = None, float('inf'), 0
    for epoch in range(1, epochs+1):
        model.train(); run=0.0
        for xb, yb in trn:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            run += loss.item()*xb.size(0)

        # Early-stopping on val loss
        model.eval(); vrun=0.0
        with torch.no_grad():
            for xb, yb in val:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = crit(logits, yb)
                vrun += loss.item()*xb.size(0)
        val_loss = vrun/len(val.dataset)

        if val_loss < best_val - 1e-7:
            best_val, best, no_imp = val_loss, model.state_dict(), 0
        else:
            no_imp += 1
            if no_imp >= patience: break

    if best: model.load_state_dict(best)
    # Return preds/probs on validation for metrics
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_valid).to(device))
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
    preds = (probs >= 0.5).astype(int)
    return model, preds, probs


def fit_rf_with_inner_cv(X_tr, y_tr) -> Tuple[RandomForestClassifier, Dict]:
    """Nested CV: tune RF on training fold only; return best estimator and params."""
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    base = RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=SEED, n_jobs=-1
    )
    gs = GridSearchCV(base, RF_PARAM_GRID, cv=inner, scoring="accuracy", n_jobs=-1)
    gs.fit(X_tr, y_tr)
    best_rf = gs.best_estimator_

    if RF_CALIBRATE:
        cal = CalibratedClassifierCV(best_rf, method="isotonic", cv=3)
        cal.fit(X_tr, y_tr)
        return cal, gs.best_params_
    else:
        return best_rf, gs.best_params_


# ========================= Feature Selection ===========================
def select_feature_indices(importances, order, top_k=None, cum_importance=None):
    if top_k is not None:
        return order[:top_k]
    if cum_importance is not None:
        cum = np.cumsum(importances[order])
        k = np.searchsorted(cum, cum_importance) + 1
        return order[:k]
    return order  # keep all

def rf_importance_for_selection(X_tr, y_tr) -> Tuple[np.ndarray, np.ndarray]:
    """Train a mid-size RF to compute importances on training fold only."""
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=2,
        max_features="sqrt", class_weight="balanced_subsample",
        n_jobs=-1, random_state=SEED
    )
    rf.fit(X_tr, y_tr)
    imps = rf.feature_importances_.astype(float)
    order = np.argsort(imps)[::-1]
    return imps, order


# ============================ Metrics ==================================
def metric_dict(y_true, y_pred, y_proba) -> Dict[str, float]:
    auc = np.nan
    if len(np.unique(y_true)) == 2:
        try:
            auc = roc_auc_score(y_true, y_proba)
        except Exception:
            auc = np.nan
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": float(auc),
    }

def summarize(metrics_list: List[Dict[str, float]]) -> Dict[str, Tuple[float,float]]:
    keys = metrics_list[0].keys()
    return {
        k: (float(np.mean([m[k] for m in metrics_list])),
            float(np.std([m[k] for m in metrics_list])))
        for k in keys
    }


# ========================== Outer CV Loop ==============================
def run_repeated_cv(df: pd.DataFrame,
                    n_splits=OUTER_N_SPLITS,
                    n_repeats=OUTER_N_REPEATS):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    # Map common label aliases, if needed
    if 'survived' not in df.columns:
        for alias in ['label', 'target', 'y']:
            if alias in df.columns:
                df = df.rename(columns={alias: 'survived'})
                break
    assert 'survived' in df.columns, "Expected label column 'survived'."

    y_all = df['survived'].astype(np.float32).values
    splitter = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=SEED
    )

    rf_metrics_all, mlp_metrics_all = [], []
    fold_id = 0

    for tr_idx, va_idx in splitter.split(np.zeros_like(y_all), y_all):
        fold_id += 1
        print(f"\n================== Outer Fold {fold_id} ==================")
        X_tr, y_tr, X_va, y_va, feat_names = features_for_indices(df, tr_idx, va_idx)

        # ----- Optional feature selection (train-only) -----
        keep_idx = np.arange(X_tr.shape[1])
        if TOP_K is not None or CUM_IMPORTANCE is not None:
            imps, order = rf_importance_for_selection(X_tr, y_tr)
            keep_idx = select_feature_indices(imps, order, TOP_K, CUM_IMPORTANCE)
            X_tr, X_va = X_tr[:, keep_idx], X_va[:, keep_idx]
            feat_names = feat_names[keep_idx]
            print(f"🧹 Using {len(keep_idx)} features via RF importance selection")

        # ----- RF: nested CV tuning on X_tr -----
        rf_best, best_params = fit_rf_with_inner_cv(X_tr, y_tr)
        if hasattr(rf_best, "predict_proba"):
            rf_probs = rf_best.predict_proba(X_va)[:, 1]
        else:
            rf_probs = rf_best.decision_function(X_va)
        rf_preds = (rf_probs >= 0.5).astype(int)
        m_rf = metric_dict(y_va, rf_preds, rf_probs)
        rf_metrics_all.append(m_rf)
        print(f"🌲 RF tuned | acc {m_rf['acc']:.4f} | bacc {m_rf['bacc']:.4f} | auc {m_rf['roc_auc']:.4f} | params {best_params}")

        # ----- MLP: single fit on X_tr -----
        _, mlp_preds, mlp_probs = train_mlp(X_tr, y_tr, X_va, y_va)
        m_mlp = metric_dict(y_va, mlp_preds, mlp_probs)
        mlp_metrics_all.append(m_mlp)
        print(f"🤖 MLP      | acc {m_mlp['acc']:.4f} | bacc {m_mlp['bacc']:.4f} | auc {m_mlp['roc_auc']:.4f}")

    # ----- Summary -----
    rf_sum = summarize(rf_metrics_all)
    mlp_sum = summarize(mlp_metrics_all)

    print("\n================== Repeated CV Summary (mean ± std) ==================")
    print(f"RF  | acc {rf_sum['acc'][0]:.4f} ± {rf_sum['acc'][1]:.4f} | "
          f"bacc {rf_sum['bacc'][0]:.4f} ± {rf_sum['bacc'][1]:.4f} | "
          f"auc {rf_sum['roc_auc'][0]:.4f} ± {rf_sum['roc_auc'][1]:.4f}")
    print(f"MLP | acc {mlp_sum['acc'][0]:.4f} ± {mlp_sum['acc'][1]:.4f} | "
          f"bacc {mlp_sum['bacc'][0]:.4f} ± {mlp_sum['bacc'][1]:.4f} | "
          f"auc {mlp_sum['roc_auc'][0]:.4f} ± {mlp_sum['roc_auc'][1]:.4f}")

    return {"rf": rf_sum, "mlp": mlp_sum, "folds": n_splits * n_repeats}


# ======================== Final Train & Testing ========================
def final_fit_and_predict(train_df: pd.DataFrame, test_df: pd.DataFrame,
                          outdir: str = OUTPUT_DIR):
    """Train final RF (with inner CV) and final MLP; generate Kaggle submissions."""
    os.makedirs(outdir, exist_ok=True)

    # --- Normalize headers & ensure label
    train_df = train_df.copy(); test_df = test_df.copy()
    train_df.columns = train_df.columns.str.strip().str.lower()
    test_df.columns  = test_df.columns.str.strip().str.lower()

    if 'survived' not in train_df.columns:
        for alias in ['label', 'target', 'y']:
            if alias in train_df.columns:
                train_df = train_df.rename(columns={alias:'survived'})
                break
    assert 'survived' in train_df.columns, "Expected 'survived' in training data."

    # --- Build train features (fit_stats from full train)
    X_all, y_all, meta = build_features(train_df, fit_stats=None)
    feat_cols_train = list(meta['feature_cols'])
    fit_stats = meta['fit_stats']

    # --- Build test features using train fit_stats, then align columns
    X_test_raw, _, meta_test = build_features(test_df, fit_stats=fit_stats)
    feat_cols_test = list(meta_test['feature_cols'])
    X_test = align_to_train_columns(X_test_raw, feat_cols_test, feat_cols_train)

    # --- Optional feature selection on full train, apply to test
    keep_idx = np.arange(X_all.shape[1])
    if TOP_K is not None or CUM_IMPORTANCE is not None:
        imps, order = rf_importance_for_selection(X_all, y_all)
        keep_idx = select_feature_indices(imps, order, TOP_K, CUM_IMPORTANCE)
        X_all = X_all[:, keep_idx]
        X_test = X_test[:, keep_idx]
        feat_cols_train = [feat_cols_train[i] for i in keep_idx]
        print(f"🧹 Final training uses {len(keep_idx)} selected features")

    # ---------------- RF: tune on full train, predict test ----------------
    rf_best, best_params = fit_rf_with_inner_cv(X_all, y_all)
    if hasattr(rf_best, "predict_proba"):
        rf_test_probs = rf_best.predict_proba(X_test)[:, 1]
    else:
        rf_test_probs = rf_best.decision_function(X_test)
    rf_test_preds = (rf_test_probs >= 0.5).astype(int)
    print(f"🌲 Final RF best params: {best_params}")

    # ---------------- MLP: train on full train (with internal holdout) ----
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
    )
    mlp_model, _, _ = train_mlp(X_tr, y_tr, X_va, y_va)
    mlp_model.eval()
    with torch.no_grad():
        logits = mlp_model(torch.tensor(X_test).to(next(mlp_model.parameters()).device))
        mlp_test_probs = torch.sigmoid(logits).cpu().numpy().ravel()
    mlp_test_preds = (mlp_test_probs >= 0.5).astype(int)

    # ---------------- Write submissions -----------------------------------
    pid_col = None
    for cand in ['passengerid', 'passenger_id', 'id']:
        if cand in test_df.columns:
            pid_col = cand; break
    if pid_col is None:
        # Fallback: create a 1..N index if Kaggle-style ID missing
        print("⚠️  PassengerId not found in test; creating a sequential Id.")
        passenger_ids = np.arange(1, X_test.shape[0]+1)
        pid_name = 'PassengerId'
    else:
        passenger_ids = test_df[pid_col].values
        pid_name = 'PassengerId'  # Kaggle expects this exact header

    def write_submission(name, preds):
        sub = pd.DataFrame({pid_name: passenger_ids, 'Survived': preds.astype(int)})
        path = os.path.join(outdir, name)
        sub.to_csv(path, index=False)
        print(f"✅ Wrote {path}")

    write_submission("submission_rf.csv", rf_test_preds)
    write_submission("submission_mlp.csv", mlp_test_preds)

    # Simple blend (average probabilities)
    blend_probs = 0.5 * rf_test_probs + 0.5 * mlp_test_probs
    blend_preds = (blend_probs >= 0.5).astype(int)
    write_submission("submission_blend.csv", blend_preds)

    return {
        "rf_params": best_params,
        "files": [
            os.path.join(outdir, "submission_rf.csv"),
            os.path.join(outdir, "submission_mlp.csv"),
            os.path.join(outdir, "submission_blend.csv"),
        ],
    }


# ============================== Main ===================================
def main(train_csv: str = DEFAULT_TRAIN, test_csv: str = DEFAULT_TEST, output_dir: str = OUTPUT_DIR):
    assert os.path.exists(train_csv), f"Missing {train_csv}"
    train_df = pd.read_csv(train_csv)
    # run CV first (optional but recommended for visibility)
    run_repeated_cv(train_df, OUTER_N_SPLITS, OUTER_N_REPEATS)

    # If test exists, build submissions
    if os.path.exists(test_csv):
        test_df = pd.read_csv(test_csv)
        final_fit_and_predict(train_df, test_df, outdir=output_dir)
    else:
        print(f"⚠️ Test file not found: {test_csv} — skipping submission files.")


if __name__ == "__main__":
    # You can also call: main(train_csv="train.csv", test_csv="test.csv", output_dir=".")
    main()



================== Outer Fold 1 ==================
🌲 RF tuned | acc 0.8268 | bacc 0.8159 | auc 0.9007 | params {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
🤖 MLP      | acc 0.8156 | bacc 0.7933 | auc 0.8728

================== Outer Fold 2 ==================
🌲 RF tuned | acc 0.8258 | bacc 0.8226 | auc 0.8905 | params {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'n_estimators': 500}
🤖 MLP      | acc 0.8090 | bacc 0.7865 | auc 0.8721

================== Outer Fold 3 ==================
🌲 RF tuned | acc 0.7921 | bacc 0.7757 | auc 0.8422 | params {'max_depth': 9, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
🤖 MLP      | acc 0.7528 | bacc 0.7270 | auc 0.8318

================== Outer Fold 4 ==================
🌲 RF tuned | acc 0.8090 | bacc 0.7977 | auc 0.8596 | params {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 300}
🤖 MLP      | acc 0.8202 | bacc 0.7956 | auc 0.8443

======